In [ ]:
# Reference: https://huggingface.co/learn/llm-course/chapter11/1

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceTB/smoltalk", "all")

In [ ]:
dataset["train"][0]

In [ ]:
# Implementation with TRL

from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer, setup_chat_format, clone_chat_template
import torch

device = "cuda"

model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model, tokenizer = setup_chat_format(model, tokenizer)

# TODO: https://huggingface.co/docs/transformers/main/en/chat_templating
# model, tokenizer, added_tokens = clone_chat_template(model, tokenizer, model_name)

# Configure trainer
training_args = SFTConfig(
    output_dir="./sft_output",
    max_steps=100,
    per_device_train_batch_size=128,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    # packing=True,  # Flash-attention is needed
    # NOTE: "messages" field is used for training by default
    #   formatting_func is used for multiple fields ('question', 'answer')
    # formatting_func=lambda s: f"### Question: {s['question']}\n ### Answer: {s['answer']}",
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

# Use "messages" field for training
trainer.train()

In [ ]:
# Implementation with TRL (LoRA)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer, setup_chat_format, clone_chat_template
import torch

from peft import PeftModel, PeftConfig, LoraConfig

device = "cuda"


model_id = "facebook/opt-350m"
model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model, tokenizer = setup_chat_format(model, tokenizer)

# peft_model_id = "ybelkada/opt-350m-lora"
# model.load_adapter(peft_model_id)

# Load in 8bit or 4bit
# peft_model_id = "ybelkada/opt-350m-lora"
# model = AutoModelForCausalLM.from_pretrained(
#     peft_model_id, quantization_config=BitsAndBytesConfig(load_in_8bit=True)
# )


# LoRA configuration
peft_config = LoraConfig(
    r=6,  # Rank dimension - typically between 4-32
    lora_alpha=8,  # LoRA scaling factor - typically 2x rank
    lora_dropout=0.05,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)
model.add_adapter(peft_config)

# Configure trainer
training_args = SFTConfig(
    output_dir="./sft_output",
    max_steps=100,
    per_device_train_batch_size=128,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    # packing=True,  # Flash-attention is needed
    # NOTE: "messages" field is used for training by default
    #   formatting_func is used for multiple fields ('question', 'answer')
    # formatting_func=lambda s: f"### Question: {s['question']}\n ### Answer: {s['answer']}",
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    peft_config=peft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

# Use "messages" field for training
trainer.train()